# PDB to ODE and NERDSS Workflow

This tutorial demonstrates the complete workflow for converting a PDB structure into NERDSS simulation files with automatic ODE and running NERDSS simulation. Here we show that ioNERDSS can take a short actin filament structure and generate assembly for a longer actin filament assembly.

## Overview

**IONERDSS.MODEL.PDB** is a pipeline that:
1. Reads protein structures from PDB/CIF files
2. Detects binding interfaces automatically
3. Generates coarse-grained molecular models
4. Calculates ODE predictions for assembly kinetics
5. Exports NERDSS simulation files for reaction-diffusion simulations

## Example: 6BNO Structure

- In this file, we'll use **6BNO** - actin filament assembly as our example.
- **6BNO** is a homo-octamer actin filament assembly. Here we show that ioNERDSS can take this short actin filament structure and generate assembly for a longer actin filament assembly.

---
## Part 1: Setup and Imports

First, import the required modules from ionerdss.

In [ ]:
#!pip install -e .. 
#  Path handling (standard library)
from pathlib import Path

# Core imports
import ionerdss as ion
from ionerdss import build_system_from_pdb

# For visualizations
import pandas as pd
import matplotlib.pyplot as plt

---
## Part 2: Configure Model Builder

### 2.1 Input Structure

Select your PDB ID and input to model builder via `source` argument. (alternatively, you can input your own PDB/CIF file path.)

#### PDB File Fetching in ionerdss

- When provided with a PDB ID (e.g., "6BNO"), ionerdss uses BioPython's PDBList class to fetch structural data from the RCSB Protein Data Bank via HTTPS (https://files.rcsb.org). The default behavior retrieves the deposited biological assembly structure in mmCIF format (e.g., 6bno.cif), which corresponds to the asymmetric unit as annotated in the PDB header.

- Importantly, ionerdss **DOES NOT** automatically fetch assembly-specific files (such as 6bno-assembly1.cif or 6bno-assembly2.cif) that may represent different biological assemblies or transformations; it retrieves only the canonical deposited structure.

- Users who need specific biological assemblies should pre-download those assembly files and provide them as local file paths rather than PDB IDs. The mmCIF format is recommended over PDB format as it contains more complete metadata and better handles large macromolecular assemblies.

### 2.2 Hyperparameters Configuration

Optional arguments of `build_system_from_pdb` allow you to control how the pipeline processes your structure.

#### Interface Detection Parameters

Interface detection parameters control how the pipeline identifies protein-protein interfaces. In the cases where an interface is not detected or an non-exist interface is detected, the user may try to adjust the following parameters.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `interface_detect_distance_cutoff` | 1.0 nm | Maximum distance between Cα atoms to consider as interface |
| `residue_cutoff` | 3 | Minimum contacting residues to validate interface |
| `residue_similarity_threshold` | 0.7 | Similarity threshold for homotypic interface detection | 

#### ProAffinity-GNN Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `predict_affinity` | `False` | Enable ProAffinity binding energy prediction |

#### Geometric Parameters

| Parameter | Default | Description | Alternative Settings |
|-----------|---------|-------------|----------------------|
| `ring_regularization_mode` | `"off"` | Regularize ring structures | `"on"` for viral capsids, `"auto"` for automatic detection |
| `steric_clash_mode` | `"off"` | Check for steric clashes | `"warn"` to detect, `"strict"` to reject clashes |

#### ODE Auto-Pipeline Parameters

`ode_enabled` controls whether to enable ODE calculation. If set to `True`, the pipeline will calculate the ODEs for the system. Make sure the adjust `ode_time_span` if the assembly is too fast or too slow.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `ode_enabled` | `False` | Enable ODE calculation | 
| `ode_time_span` | `(0.0, 10.0)` | Simulation time range (s) |
| `ode_solver_method` | `"BDF"` | ODE solver algorithm | 
| `ode_atol` | `1e-6` | Absolute tolerance |
| `ode_plot` | `True` | Generate concentration plots |
| `ode_save_csv` | `True` | Save time-series data | 
| `ode_initial_concentrations` | `None` | Custom initial conditions |

#### Transition Matrix Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `count_transition` | `False` | Enable transition matrix tracking |
| `transition_matrix_size` | `100` | Size of matrix (max cluster size expected) |
| `transition_write` | `1000` | Write every 1000 iterations |

---
## Part 3: Build the System

The `build_system()` method orchestrates the entire pipeline:

1. **Parse PDB** - Read structure from file
2. **Detect Interfaces** - Find binding sites between chains
3. **Coarse-grain** - Create simplified molecular representations
4. **Build Templates** - Generate reaction templates
5. **Construct System** - Assemble complete molecular system
6. **Export NERDSS** - Write simulation input files
7. **Run ODE**  - Calculate assembly kinetics 
8. **Save Outputs** - Store results

All of this happens automatically:

In [ ]:
pdb_id = "6bno"

# Build the system using simplified API
# This should take ~10 seconds for 6bno
system = build_system_from_pdb(
    source=pdb_id,
    workspace_path=f"{pdb_id}_dir",
    # Interface detection
    #interface_detect_distance_cutoff=1.0,  # Standard cutoff for protein interfaces
    interface_detect_distance_cutoff=1.5,
    interface_detect_n_residue_cutoff=5,
    chain_grouping_seq_threshold=0.5,
    ring_regularization_mode="off",
    nerdss_water_box=[500.0, 500.0, 500.0],
    
    # ODE Pipeline Configuration
    ode_enabled=True,            # Now using System-compatible generator!
    ode_time_span=None,   # Auto-calculated based on NERDSS simulation time
    ode_solver_method="BDF",     # Solver for stiff systems
    ode_plot=True,               # Generate plots
    ode_save_csv=True,            # Save data to CSV

    # Transition matrix parameters
    #count_transition=True,        # Enable transition matrix tracking
    transition_matrix_size=10,   # Size of matrix (max cluster size expected)
    transition_write=1000,         # Write every 1000 iterations
)

#### (Optional) 3.1. Visualize the Coarse-grained System with PyMOL

A `.pml` file is generated in the `visualizations` directory. You can open it with PyMOL to visualize the coarse-grained system. Useful commands for visualization:

- `hide labels`: hide the bond length labels
- `util.cbc`: color by chain

---
## Part 4: (Optional) Analyze ODE Results

The ODE auto-pipeline generates predictions for assembly kinetics. Let's examine the results.

### 4.1 Load ODE Data

In [ ]:
# Load the ODE solution CSV
ode_csv_path = f"./{pdb_id}_dir/ode_results/ode_solution_simple.csv"
ode_data = pd.read_csv(ode_csv_path)

# Show the first 10 rows of dataframe
ode_data.head(10)

### 4.2 Visualize Assembly Kinetics

In [ ]:
# import the display and Image functions from IPython.display
from IPython.display import display, Image

# Specify the path to your PNG file
image_path = f'./{pdb_id}_dir/ode_results/ode_solution_simple.png'

# Display the image
display(Image(filename=image_path))


---
## Part 5: (Optional) Review Generated Files

The pipeline creates several output files for further analysis and simulation.

In [ ]:
# List all generated files
workspace_path = Path(f"{pdb_id}_dir")

print("Generated Files:")
print("\n ODE Results:")
ode_dir = workspace_path / "ode_results"
if ode_dir.exists():
    for file in sorted(ode_dir.glob("*")):
        size = file.stat().st_size / 1024  # KB
        print(f"  {file.name:<30} ({size:>6.1f} KB)")

print("\n NERDSS Input Files:")
nerdss_dir = workspace_path / "nerdss_files"
if nerdss_dir.exists():
    for file in sorted(nerdss_dir.glob("*.mol")) + sorted(nerdss_dir.glob("*.inp")):
        size = file.stat().st_size / 1024  # KB
        print(f"  {file.name:<30} ({size:>6.1f} KB)")

print("\n System Data:")
outputs_dir = workspace_path / "outputs" / "systems"
if outputs_dir.exists():
    for file in sorted(outputs_dir.glob("*.json")):
        size = file.stat().st_size / 1024  # KB
        print(f"  {file.name:<30} ({size:>6.1f} KB)")

---
## Part 6: Run NERDSS Simulation

There are two options to run NERDSS:

1. Manually run NERDSS with python subprocess
2. Use the NERDSS auto-pipeline

### Option 1: Manually run NERDSS with python subprocess

If you have NERDSS installed, you can run NERDSS simulations with by calling the NERDSS executable with python subprocess.

**Note**: This requires NERDSS to be installed on your system. The user also has to specify the path to the NERDSS executable.

In [ ]:
# run NERDSS with subprocess
import subprocess

# Check if NERDSS is available
# nerdss_cmd should be replaced with the actual path to the NERDSS executable
nerdss_cmd = "~/Workspace/Reaction_ode/nerdss_development/bin/nerdss"
nerdss_path = Path(nerdss_cmd).expanduser() # replaces tilde with appropriate user home path

if nerdss_path.exists():
    
    # Run NERDSS
    result = subprocess.run(
        f"{nerdss_cmd} -f parms.inp",
        shell=True,
        cwd=f"{pdb_id}_dir/nerdss_files",
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("✓ NERDSS simulation completed!")
        print(f"\nCheck {pdb_id}_dir/nerdss_files/ for output files")
    else:
        print("⚠ NERDSS simulation failed")
        print(result.stderr[:500])
else:
    print("⚠ NERDSS not found at:", nerdss_cmd)

---
## (Optional) Part 7: Analyze NERDSS Output

After running NERDSS simulations, we can analyze the results using the `Analyzer` class.


In [ ]:
# Initialize Analyzer with NERDSS output directory
analysis = ion.Analyzer("6bno_dir")

# Display discovered simulations
print(f"Found {len(analysis.simulations)} simulation(s)")
for i, sim in enumerate(analysis.simulations):
    print(f"  [{i}] Simulation ID: {sim.id}")

#################
# Plot the NERDSS trajectory alone
plt.figure()

sim = analysis.get_simulation(0)
complex_compositions = [{"A":n} for n in range(1,11)] # A1 through A10

# get the time series data for the above complexes
time, counts = sim.get_time_series(complex_compositions)

# plot all the returned data
for i in range(10):
    plt.plot(time,counts[i],label=str(complex_compositions[i]))

plt.legend()
plt.show()

################
# plot ODE and NERDSS side to side 
# Specify the path to your csv file
csv_path = f'./{pdb_id}_dir/ode_results/ode_solution_simple.csv'

# Display the image
df = pd.read_csv(csv_path)

# Create a new figure
plt.figure()

# Only plot first n species
plot_first_n_species = 4

# plot nerdss simulation
for i in range(plot_first_n_species):
    plt.plot(time, counts[i], label=str(f"A{i+1}(sim)"))

# ode results
for col in df.columns:
    # plot first 
    if col != "time":
        # df contains concentration in uM, convert to counts
        # 1 uM ~= 75 in 500 nm x 500 nm x 500 nm box
        # Change this if you used a different initial count
        initial_count_number = 75
        plt.plot(df["time"], df[col] * initial_count_number,
            label = (f"{col}(ode)"), ls = "--")

plt.legend()
plt.show()

In [ ]:
# Initialize Analyzer with NERDSS output directory
analysis = ion.Analyzer("6bno_dir")

# Display discovered simulations
print(f"Found {len(analysis.simulations)} simulation(s)")
for i, sim in enumerate(analysis.simulations):
    print(f"  [{i}] Simulation ID: {sim.id}")

# Create a figure with multiple subplots to show different analyses
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Size distribution
analysis.plot.size_distribution(simulation_index=0, ax=axes[0, 0])
axes[0, 0].set_title('Cluster Size Distribution', fontweight='bold')

# Plot 2: Free energy profile
analysis.plot.free_energy(simulation_index=0, ax=axes[0, 1])
axes[0, 1].set_title('Free Energy Profile', fontweight='bold')

# Plot 3: Growth vs shrinkage probabilities
analysis.plot.transitions(simulation_index=0, ax=axes[1, 0])
axes[1, 0].set_title('Assembly Transitions', fontweight='bold')

# Plot 4: Transition matrix heatmap
analysis.plot.heatmap(simulation_index=0, ax=axes[1, 1])
axes[1, 1].set_title('Transition Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig('6bno_dir/nerdss_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary and Next Steps

### What We've Done

 Loaded 6BNO structure  
 Configured hyperparameters  
 Detected binding interfaces automatically  
 Generated ODE model with all subcomplexes (A → A₈)  
 Exported NERDSS simulation files
 Ran NERDSS simulation for the assembly of the filament extending beyond the PDB file
 Visualized assembly kinetics  

---

*Tutorial created: 2026-2-4*  
*IONERDSS Version: 1.2.0*